# 🧪 Credit Risk & Loan Default Prediction: 03 - Model Experimentation & Benchmark

### Objective:
Compare multiple classification algorithms using Stratified K-Fold Cross-Validation, measuring ROC-AUC, PR-AUC (Average Precision), Recall, and F1 Score under severe class imbalance.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import sys
sys.path.append('..')
from src.features.feature_engineering import FeatureEngineer, CatBoostPreprocessor


## 1. Load Data & Prepare Cross-Validation


In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns='TARGET')
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}


## 2. Define Model Candidates


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1, verbosity=-1),
    'CatBoost': CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, loss_function='Logloss', eval_metric='AUC', random_seed=42, verbose=0)
}
print(f'Evaluating {len(models)} algorithms...')
